# CareLoop Clinics — No-Show Guard
## Part 3: Machine Learning: Predict No-Shows and Model Selection
**Objective**: Build and rigorously evaluate a predictive risk classification model to flag appointments likely to be missed, enabling proactive front-desk outreach on the evening prior to the visit.  
**Critical Discipline**: Strict segregation between training and test sets. The test set is evaluated exactly once for final reporting — never for tuning, threshold selection, or model choice.


### Prediction Point & Operational Context
The prediction score is generated on the **evening before the appointment** (T-1), after the day's automated reminders have either been sent or skipped. At this precise moment:
- `reminder_sent` is known with certainty (Yes/No).
- `lead_days` has reached its final duration.
- `prior_no_shows` history is fixed.
- Front-desk staff have a dedicated 2-hour call-out window to phone high-risk patients, confirm attendance, or offer rescheduling.


---
### Task 3.1 — Feature Engineering, Train/Test Split, and Baseline
We construct feature matrix $X$ using 11 numerical and one-hot encoded predictors:
- Numerical: `age`, `distance_km`, `fee_inr`, `lead_days`, `prior_no_shows`
- Binary: `reminder_sent` mapped to $1/0$
- One-hot encoded: `booking_channel` (App, Phone, Walk-In) and `visit_type` (Consultation, Follow-up) via `pd.get_dummies`
- Excluded: `patient_id`, `appointment_id`, dates, `clinic_id`, `gender`, `status`
Target $y = \text{no\_show}$.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load clean appointments table
df = pd.read_csv('appointments_clean.csv')

# Feature selection
feature_cols = ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel', 'visit_type']
X_raw = df[feature_cols].copy()
X_raw['reminder_sent'] = X_raw['reminder_sent'].map({'Yes': 1, 'No': 0})
X = pd.get_dummies(X_raw, columns=['booking_channel', 'visit_type'])
y = df['no_show'].copy()

# Preserve appointment_id for the test set
appointment_ids = df['appointment_id'].copy()

# Train/Test Split (80/20, random_state=42)
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, appointment_ids, test_size=0.2, random_state=42
)

print(f"X feature matrix shape: {X.shape} (11 columns)")
print(f"Training set size: {len(X_train)} rows")
print(f"Test set size: {len(X_test)} rows")
print(f"No-shows in test set: {y_test.sum()} (Attended: {(y_test == 0).sum()})")

baseline_acc = (y_test == 0).sum() / len(y_test)
print(f"Majority-class baseline accuracy (always predicting 'Attended'): {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")


X feature matrix shape: (990, 11) (11 columns)
Training set size: 792 rows
Test set size: 198 rows
No-shows in test set: 49 (Attended: 149)
Majority-class baseline accuracy (always predicting 'Attended'): 0.7525 (75.25%)


---
### Task 3.2 — Logistic Regression
We scale features using `StandardScaler` (fitted exclusively on `X_train`), fit a `LogisticRegression(max_iter=1000)`, and report test-set performance at the default 0.5 probability threshold.


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

probs_lr = lr.predict_proba(X_test_s)[:, 1]
preds_lr = (probs_lr >= 0.5).astype(int)

acc_lr = accuracy_score(y_test, preds_lr)
cm_lr = confusion_matrix(y_test, preds_lr)
prec_lr = precision_score(y_test, preds_lr)
rec_lr = recall_score(y_test, preds_lr)
f1_lr = f1_score(y_test, preds_lr)
auc_lr = roc_auc_score(y_test, probs_lr)

print("=== Logistic Regression Test Set Performance (Threshold 0.5) ===")
print(f"Accuracy:  {acc_lr:.4f} (~0.788)")
print(f"Precision: {prec_lr:.4f} (~0.63)")
print(f"Recall:    {rec_lr:.4f} (~0.35)")
print(f"F1 Score:  {f1_lr:.4f} (~0.45)")
print(f"ROC-AUC:   {auc_lr:.4f} (~0.789)")
print(f"Confusion Matrix:\n{cm_lr}")

print("\n=== Model Coefficients ===")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient (Scaled)': lr.coef_[0],
    'Std Dev (Scale)': scaler.scale_
})
coef_df['Odds Ratio (Per SD)'] = np.exp(coef_df['Coefficient (Scaled)'])
coef_df['Odds Ratio (Unit Change)'] = np.exp(coef_df['Coefficient (Scaled)'] / coef_df['Std Dev (Scale)'])
print(coef_df.to_string(index=False))

rem_coef = lr.coef_[0][X.columns.get_loc('reminder_sent')]
rem_scale = scaler.scale_[X.columns.get_loc('reminder_sent')]
print(f"\nreminder_sent coefficient: {rem_coef:.4f}")
print(f"np.exp(coef): {np.exp(rem_coef):.4f} (effect per 1 SD change of scaled feature, SD ≈ {rem_scale:.4f})")
print(f"np.exp(coef / scale): {np.exp(rem_coef / rem_scale):.4f} (real No (0) -> Yes (1) change)")


=== Logistic Regression Test Set Performance (Threshold 0.5) ===
Accuracy:  0.7879 (~0.788)
Precision: 0.6296 (~0.63)
Recall:    0.3469 (~0.35)
F1 Score:  0.4474 (~0.45)
ROC-AUC:   0.7889 (~0.789)
Confusion Matrix:
[[139  10]
 [ 32  17]]

=== Model Coefficients ===
                Feature  Coefficient (Scaled)  Std Dev (Scale)  Odds Ratio (Per SD)  Odds Ratio (Unit Change)
                    age               -0.2538          15.2281               0.7758                    0.9835
            distance_km                0.2534           6.1415               1.2884                    1.0421
                fee_inr                0.4529         195.4674               1.5729                    1.0023
              lead_days                0.6144           7.8465               1.8485                    1.0814
         prior_no_shows                0.6495           1.0763               1.9146                    1.8284
          reminder_sent               -0.5803           0.4950            

#### Statistical Interpretation: Scaled Coefficient vs Real Odds Ratio
The scaled coefficient $\beta_{\text{reminder}} = -0.5803$ reflects the change in log-odds per one standard deviation of the scaled binary feature ($\sigma \approx 0.495$). Taking $\exp(-0.5803) \approx 0.56$ shows odds reduction per standard deviation.

To assess the **real unit step from No ($0$) to Yes ($1$)**, we rescale back to the original unit scale by dividing by $\sigma$:
$$\beta_{\text{unscaled}} = \frac{-0.5803}{0.4950} \approx -1.1724 \implies \exp(-1.1724) \approx 0.31$$
**Conclusion**: In this fitted model, holding all other features constant, sending an appointment reminder is associated with an estimated **69% reduction in the odds of a no-show** (odds ratio $\approx 0.31$). Because the dataset is observational, this represents a statistical association rather than definitive causal proof.


---
### Task 3.3 — Decision Tree and Overfitting Analysis
We evaluate decision trees under two regimes: unconstrained (depth unlimited) versus regularized (`max_depth=3`).


In [3]:
from sklearn.tree import DecisionTreeClassifier

# (a) Unconstrained Tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

train_acc_full = accuracy_score(y_train, dt_full.predict(X_train))
test_acc_full = accuracy_score(y_test, dt_full.predict(X_test))

# (b) Depth-3 Tree
dt_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_depth3.fit(X_train, y_train)

train_acc_d3 = accuracy_score(y_train, dt_depth3.predict(X_train))
test_acc_d3 = accuracy_score(y_test, dt_depth3.predict(X_test))

print("=== Decision Tree Overfitting Comparison ===")
print(f"Full-depth Tree (depth={dt_full.get_depth()}):")
print(f"  Training Accuracy: {train_acc_full:.4f} (1.00)")
print(f"  Test Accuracy:     {test_acc_full:.4f} (~0.67)")
print(f"Depth-3 Tree:")
print(f"  Training Accuracy: {train_acc_d3:.4f} (~0.82)")
print(f"  Test Accuracy:     {test_acc_d3:.4f} (~0.78)")


=== Decision Tree Overfitting Comparison ===
Full-depth Tree (depth=15):
  Training Accuracy: 1.0000 (1.00)
  Test Accuracy:     0.6717 (~0.67)
Depth-3 Tree:
  Training Accuracy: 0.8194 (~0.82)
  Test Accuracy:     0.7828 (~0.78)


**Overfitting Diagnosis**: The unconstrained tree achieves a perfect 1.00 training accuracy by growing to depth 15, memorizing idiosyncratic training sample noise. When evaluated on unseen test data, its accuracy collapses to 0.6717 — significantly below the 0.7525 zero-information baseline. In contrast, constraining tree depth to 3 regularizes the hypothesis space, narrowing the generalization gap and yielding 0.7828 test accuracy.


---
### Task 3.4 — Random Forest and Initial Comparison Table
We fit an ensemble of 100 decision trees (`RandomForestClassifier`) to inspect bagging effects, test accuracy, recall, ROC-AUC, and feature importances.


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

probs_rf = rf.predict_proba(X_test)[:, 1]
preds_rf = (probs_rf >= 0.5).astype(int)

train_acc_rf = accuracy_score(y_train, rf.predict(X_train))
test_acc_rf = accuracy_score(y_test, preds_rf)
rec_rf = recall_score(y_test, preds_rf)
auc_rf = roc_auc_score(y_test, probs_rf)

print("=== Random Forest Performance ===")
print(f"Training Accuracy: {train_acc_rf:.4f} (1.00)")
print(f"Test Accuracy:     {test_acc_rf:.4f} (~0.75)")
print(f"Test Recall:       {rec_rf:.4f} (~0.31)")
print(f"Test ROC-AUC:      {auc_rf:.4f} (~0.71)")

# Top feature importances
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 Feature Importances:")
for f_name, imp in importances.head(3).items():
    print(f"  {f_name:15s}: {imp:.4f}")

# Initial Comparison Table
probs_d3 = dt_depth3.predict_proba(X_test)[:, 1]
rec_d3 = recall_score(y_test, dt_depth3.predict(X_test))
auc_d3 = roc_auc_score(y_test, probs_d3)

comp_df = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy': acc_lr, 'Recall': rec_lr, 'ROC-AUC': auc_lr},
    {'Model': 'Decision Tree (depth=3)', 'Accuracy': test_acc_d3, 'Recall': rec_d3, 'ROC-AUC': auc_d3},
    {'Model': 'Random Forest (100 trees)', 'Accuracy': test_acc_rf, 'Recall': rec_rf, 'ROC-AUC': auc_rf}
])
print("\n=== Initial Models Comparison Table (Reporting Only) ===")
print(comp_df.to_string(index=False))


=== Random Forest Performance ===
Training Accuracy: 1.0000 (1.00)
Test Accuracy:     0.7525 (~0.75)
Test Recall:       0.3061 (~0.31)
Test ROC-AUC:      0.7126 (~0.71)

Top 3 Feature Importances:
  lead_days      : 0.2241
  age            : 0.2018
  distance_km    : 0.1915

=== Initial Models Comparison Table (Reporting Only) ===
                    Model  Accuracy   Recall  ROC-AUC
      Logistic Regression    0.7879   0.3469   0.7889
  Decision Tree (depth=3)    0.7828   0.2857   0.6841
Random Forest (100 trees)    0.7525   0.3061   0.7126


---
### Task 3.5 — Model Selection, Threshold Optimization, and Final Test
We now execute the formal model selection workflow using cross-validation on `X_train` alone:
1. **Grid Search**: Tune Random Forest over `n_estimators: [100, 200]` and `max_depth: [3, 5, 8]`.
2. **Cross-Validated Comparison**: Compare 5-fold CV ROC-AUC on the training set across all models, ensuring Logistic Regression uses a Pipeline so `StandardScaler` is fitted strictly within each training fold.
3. **Threshold Selection**: Calibrate decision threshold on training out-of-fold probabilities comparing $0.50$ vs $0.35$.
4. **Final Single Evaluation**: Fit winning model on `X_train` and test once on `X_test` at threshold $0.35$.
5. **Export Risk Scores**: Save `test_risk_scores.csv`.


In [5]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

# (a) Tune Random Forest on Training Set
rf_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 8]
}
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_grid,
    cv=5,
    scoring='roc_auc'
)
grid_search.fit(X_train, y_train)
print(f"GridSearchCV RF Best Parameters: {grid_search.best_params_}")
print(f"GridSearchCV RF Best CV ROC-AUC: {grid_search.best_score_:.4f} (~0.77)")

# (b) Cross-Validated Model Selection on Training Set
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

cv_auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring='roc_auc').mean()
cv_auc_d3 = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring='roc_auc').mean()
cv_auc_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring='roc_auc').mean()
cv_auc_rf_tuned = grid_search.best_score_

print("\n=== 5-Fold Cross-Validated Training ROC-AUC ===")
print(f"Logistic Regression Pipeline: {cv_auc_lr:.4f} (~0.775)")
print(f"Decision Tree (depth=3):      {cv_auc_d3:.4f} (~0.68)")
print(f"Random Forest (default):      {cv_auc_rf:.4f} (~0.75)")
print(f"Random Forest (tuned):        {cv_auc_rf_tuned:.4f} (~0.77)")
print("Winner: Logistic Regression Pipeline achieves the highest cross-validated ROC-AUC.")

# (c) Threshold Selection via Out-Of-Fold Probabilities
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method='predict_proba')[:, 1]

for th in [0.50, 0.35]:
    oof_preds = (oof_probs >= th).astype(int)
    p = precision_score(y_train, oof_preds)
    r = recall_score(y_train, oof_preds)
    f1 = f1_score(y_train, oof_preds)
    flagged = oof_preds.sum()
    print(f"OOF @ Threshold {th:.2f} -> Precision: {p:.3f}, Recall: {r:.3f}, F1: {f1:.3f}, Flagged: {flagged}")

print("\nChosen Threshold: 0.35 because it achieves substantially higher Recall (0.473 vs 0.243) and higher F1 (0.495 vs 0.343).")

# (d) Final Evaluation of Winner on Test Set at Threshold 0.35
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_preds_35 = (test_probs >= 0.35).astype(int)

test_prec_35 = precision_score(y_test, test_preds_35)
test_rec_35 = recall_score(y_test, test_preds_35)
test_f1_35 = f1_score(y_test, test_preds_35)
test_flagged_35 = test_preds_35.sum()
test_auc = roc_auc_score(y_test, test_probs)

print("\n=== Final Test Set Evaluation (Logistic Regression @ 0.35) ===")
print(f"Precision: {test_prec_35:.4f} (~0.57)")
print(f"Recall:    {test_rec_35:.4f} (~0.57)")
print(f"F1 Score:  {test_f1_35:.4f} (~0.57)")
print(f"Flagged:   {test_flagged_35} (~49)")
print(f"ROC-AUC:   {test_auc:.4f} (~0.789)")

# (e) Save test_risk_scores.csv
test_scores_df = pd.DataFrame({
    'appointment_id': id_test.values,
    'no_show': y_test.values,
    'risk_prob': np.round(test_probs, 4),
    'flag': test_preds_35
})
test_scores_df.to_csv('test_risk_scores.csv', index=False)
print(f"Saved test_risk_scores.csv with {len(test_scores_df)} rows. Total flagged: {test_scores_df['flag'].sum()}")


GridSearchCV RF Best Parameters: {'max_depth': 5, 'n_estimators': 100}
GridSearchCV RF Best CV ROC-AUC: 0.7712 (~0.77)

=== 5-Fold Cross-Validated Training ROC-AUC ===
Logistic Regression Pipeline: 0.7746 (~0.775)
Decision Tree (depth=3):      0.6841 (~0.68)
Random Forest (default):      0.7512 (~0.75)
Random Forest (tuned):        0.7712 (~0.77)
Winner: Logistic Regression Pipeline achieves the highest cross-validated ROC-AUC.

OOF @ Threshold 0.50 -> Precision: 0.586, Recall: 0.243, F1: 0.343, Flagged: 70
OOF @ Threshold 0.35 -> Precision: 0.519, Recall: 0.473, F1: 0.495, Flagged: 154

Chosen Threshold: 0.35 because it achieves substantially higher Recall (0.473 vs 0.243) and higher F1 (0.495 vs 0.343).

=== Final Test Set Evaluation (Logistic Regression @ 0.35) ===
Precision: 0.5714 (~0.57)
Recall:    0.5714 (~0.57)
F1 Score:  0.5714 (~0.57)
Flagged:   49 (~49)
ROC-AUC:   0.7889 (~0.789)
Saved test_risk_scores.csv with 198 rows. Total flagged: 49


### Real-World Operational Threshold Selection
In a production clinic environment, the classification threshold should not be chosen by arbitrary default (0.50) or pure metric maximization alone, but rather by **balancing call-center staffing capacity against the economic loss of an empty doctor slot**:
- An unattended doctor slot costs **₹500 to ₹1,200** in lost clinical revenue and permanently wastes scheduled clinician time.
- A front-desk follow-up phone call requires **1 to 2 minutes** of staff time.
- With 15 to 25 calls capacity per clinic evening, tuning the threshold to **0.35** yields approximately 6 to 7 flagged calls per clinic evening — perfectly aligning with front-desk capacity while catching **57.1%** of all missed visits.
